# 🧭 生成式AI應用開發｜第04週

## Prompt Engineering 實務：把模糊需求變成可測試的指令

本週延續第 3 週 OpenAI Responses API，透過摘要、分類、改寫與依據資料問答，練習設計、比較及測試 prompt。


> **教師版**：包含完整參考答案與課堂示範；建議先讓學生完成練習，再分段講解。


## 🎯 本週學習目標

完成本週後，你應該能夠：

1. 分辨 `instructions` 與使用者 `input` 的責任。
2. 用角色、任務、背景、限制、輸出格式寫出清楚的 prompt。
3. 使用 Markdown 標題與 XML 標籤區隔指令、範例和資料。
4. 用任務拆解與 few-shot examples 改善結果一致性。
5. 建立可調整參數的摘要、分類、改寫、問答四種小功能。
6. 用 `compare_prompts()`、固定案例與程式檢查比較 prompt。

<div style="border-left:5px solid #1a73e8;padding:10px 14px;background:#eef5ff;">
<font color="#174ea6"><b>核心觀念</b></font>：Prompt Engineering 是把需求寫成可理解、可重複、可測試的規格。
</div>


## 🗺️ 三小時實作流程

| 時間 | 主題 | 實作成果 |
|---:|---|---|
| 0:00–0:25 | 基準 prompt 與五個元素 | 比較模糊／明確 prompt |
| 0:25–1:05 | instructions、分隔符、任務拆解 | 建立可重用模板 |
| 1:05–1:35 | zero-shot 與 few-shot | 完成文字分類器 |
| 1:35–2:20 | 四種應用任務 | 摘要、分類、改寫、問答 |
| 2:20–2:45 | Prompt 測試 | 測試案例與檢查規則 |
| 2:45–3:00 | 綜合練習與回顧 | 個人 prompt 小工具 |


---

## 1️⃣ 環境準備

沿用第 3 週設定：API key 放在 Colab Secrets 的 `OPENAI_API_KEY`，不要直接寫進 Notebook。程式會先檢查金鑰，再建立 API client；若設定不完整，會提供明確錯誤訊息。


In [ ]:
# 安裝或更新 OpenAI Python SDK
!pip install -q --upgrade openai


In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境時，沿用作業系統環境變數。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "找不到 OPENAI_API_KEY。請先設定 Colab Secret 或本機環境變數，再重新執行此 cell。"
    )

# 課堂成本預設；可在 OPENAI_MODEL 環境變數中覆蓋。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI()
print("本週模型：", MODEL)
print("API key 已設定（不顯示內容）")


In [ ]:
def ask_ai(user_input, instructions="你是一位清楚、可靠的繁體中文助理"):
    """送出一次 Responses API 請求；失敗時提供明確錯誤，不把錯誤當成模型輸出。"""
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")
    if not isinstance(instructions, str) or not instructions.strip():
        raise ValueError("instructions 必須是非空白字串。")

    try:
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=user_input,
        )
    except Exception as exc:
        raise RuntimeError(f"OpenAI API 呼叫失敗：{exc}") from exc
    return response.output_text


def compare_prompts(user_input, prompt_variants):
    """用同一輸入執行多組 instructions；每個 variant 會產生一次 API 呼叫。"""
    if not isinstance(prompt_variants, dict) or not prompt_variants:
        raise ValueError("prompt_variants 必須是非空字典。")
    return {
        name: ask_ai(user_input, instructions)
        for name, instructions in prompt_variants.items()
    }


<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>成本提醒</b></font>：每次執行 `ask_ai(...)` 都會送出 API 請求；`compare_prompts()` 與測試迴圈會依版本數、案例數呼叫多次。批次測試預設關閉，確認成本後再開啟。
</div>

> **可重現性提醒**：課堂可用模型 alias 方便操作；正式評估時應記錄模型名稱，必要時固定模型 snapshot，並用同一批測試案例重跑。


---

## 2️⃣ 從模糊 prompt 建立基準結果

先保留簡單版本作為 baseline。之後每次只調整一個主要變因，才能判斷是哪項改動造成差異。


In [ ]:
course_intro = """
生成式AI應用開發是一門大三課程，學生會使用 Python、OpenAI API、
Streamlit 與 GitHub 完成 AI 應用。課程重視實作、API key 安全、成本控制，
並以期中個人專題和期末小組專題整合所學。
"""

comparison_input = f"""請摘要以下課程資料：
<course>
{course_intro}
</course>"""
baseline_instructions = "請摘要使用者提供的課程資料。"
baseline_result = ask_ai(comparison_input, baseline_instructions)
print(baseline_result)


### ✍️ 觀察紀錄

- 摘要有幾句？是否使用繁體中文？
- 是否保留課程工具與評量方式？
- 重跑一次時，長度與格式是否相同？

模糊 prompt 不一定失敗，但它沒有先定義「什麼叫成功」。


---

## 3️⃣ Prompt 的五個實用元素

| 元素 | 要回答的問題 | 範例 |
|---|---|---|
| 角色 Role | 模型以什麼身份工作？ | 大學課程助教 |
| 任務 Task | 具體完成什麼？ | 摘要課程介紹 |
| 背景 Context | 必要資料是什麼？ | `<course>...</course>` |
| 限制 Constraints | 長度、語言、禁忌？ | 60 字內、不新增資訊 |
| 輸出 Output | 結果長什麼樣？ | 一句總結＋三個重點 |

口訣：**誰、做什麼、根據什麼、遵守什麼、輸出什麼。**


In [ ]:
clear_instructions = """
# Identity
你是大學課程的教學助理，擅長整理繁體中文課程資訊。

# Instructions
1. 只能根據使用者提供的課程文字摘要，不得新增資訊。
2. 先寫一句 60 字以內的總結。
3. 再列三個重點，涵蓋技術、課程特色與評量。
4. 使用繁體中文。
"""
clear_result = ask_ai(comparison_input, clear_instructions)
print(clear_result)

# 同一輸入比較多組 instructions。改成 True 會再送出 2 次 API 請求。
RUN_PROMPT_COMPARISON = False
if RUN_PROMPT_COMPARISON:
    comparison_results = compare_prompts(
        comparison_input,
        {"Baseline": baseline_instructions, "明確版本": clear_instructions},
    )
    for name, result in comparison_results.items():
        print(f"===== {name} =====")
        print(result)


### 🔍 比較 baseline 與明確版本

| 比較面向 | Baseline | 明確版本 |
|---|---|---|
| 長度是否可預期 | 通常不穩定 | 有 60 字總結與三點要求 |
| 格式是否一致 | 未指定 | 總結＋三個重點 |
| 是否保留必要資訊 | 可能遺漏 | 明定技術、特色、評量 |
| 是否出現額外推測 | 沒有禁止 | 明確禁止新增資訊 |

**教師觀察提示**：結果不必逐字相同。請引導學生先定義成功條件，再比較相同輸入的不同 instructions；需要投影並排結果時，開啟上一格的 `RUN_PROMPT_COMPARISON`。


---

## 4️⃣ `instructions` 與 `input`：規則和資料分開

- `instructions`：應用程式提供的固定角色、規則、流程與範例。
- `input`：每次請求會改變的使用者任務與資料。

課堂常把固定高層規則泛稱為 system prompt；本課使用 Responses API，因此用 `instructions` 實作。可把固定規則想成函式定義，把每次輸入想成函式參數。


In [ ]:
def summarize_text(text, max_chars_per_point=35, num_points=3):
    if max_chars_per_point <= 0 or num_points <= 0:
        raise ValueError("字數與重點數必須是正整數。")

    instructions = f"""
# Identity
你是精準的繁體中文摘要助理。
# Instructions
- 只根據提供的文字作答，不新增內容。
- 保留重要名稱、數字與限制。
- 輸出一個標題與 {num_points} 個條列重點。
- 每點不超過 {max_chars_per_point} 字。
"""
    user_input = f"""請摘要：
<source_text>
{text}
</source_text>"""
    return ask_ai(user_input, instructions)


print(summarize_text(course_intro, max_chars_per_point=35, num_points=3))


### 🧱 Markdown 與 XML 分隔資料

- Markdown 標題讓 prompt 層次清楚、容易維護。
- XML 標籤指出資料與範例的開始、結束。
- 分隔符能降低規則與資料混淆，但不是完整安全機制；外部文字仍是不可信資料。


---

## 5️⃣ 任務拆解：把複雜要求排成步驟

當任務包含多個判斷，明確寫出可觀察的工作流程，通常比只寫「仔細思考」更容易測試。


In [ ]:
feedback = "介面很漂亮，但註冊流程一直失敗，客服兩天後才回覆。"
analysis_instructions = """
# Identity
你是產品回饋分析助理。
# Workflow
1. 找出稱讚項目。2. 找出問題。3. 判斷最優先問題。4. 提出一項改善建議。
# Output format
- 優點：
- 問題：
- 優先處理：
- 建議：
"""
print(ask_ai(f"<feedback>{feedback}</feedback>", analysis_instructions))


---

## 6️⃣ Zero-shot 與 Few-shot

- **Zero-shot**：描述規則，不提供答案範例。
- **Few-shot**：提供少量且具代表性的輸入／輸出範例，讓模型掌握標籤、格式與邊界。

範例應涵蓋多樣情況；太相似的範例無法清楚呈現分類界線。


In [ ]:
COURSE_CATEGORIES = ["內容", "進度", "設備", "其他"]
COURSE_EXAMPLES = [
    ("希望 API 範例多一點。", "內容"),
    ("操作速度太快。", "進度"),
    ("教室電腦無法安裝套件。", "設備"),
    ("希望下次分組討論。", "其他"),
]


def classify_text(text, categories, examples=None):
    if not categories or len(categories) != len(set(categories)):
        raise ValueError("categories 必須包含不重複的分類名稱。")

    category_text = "、".join(f"「{name}」" for name in categories)
    example_text = ""
    for sample_input, sample_output in examples or []:
        example_text += (
            f"<example><input>{sample_input}</input>"
            f"<output>{sample_output}</output></example>\n"
        )

    instructions = f"""
# Identity
你是文字分類器。
# Instructions
- 分類為 {category_text} 其中一類。
- 只輸出一個分類名稱，不要解釋。
# Examples
{example_text}
"""
    result = ask_ai(f"<text_to_classify>{text}</text_to_classify>", instructions).strip()
    if result not in categories:
        raise ValueError(f"模型輸出不符合分類契約：{result!r}")
    return result


def classify_feedback(text):
    return classify_text(text, COURSE_CATEGORIES, COURSE_EXAMPLES)


for sample in ["JSON 希望再示範一次。", "投影畫面太小。", "練習時間不夠。"]:
    print(sample, "→", classify_feedback(sample))


<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>先不要混淆</b></font>：本週只用文字規則限制輸出，仍可能出現意外格式；第 6 週再用 Structured Outputs 與 schema 做程式層級保證。
</div>


---

## 7️⃣ 四種常見 Prompt 應用

### A. 摘要
前面的 `summarize_text()` 示範目標讀者、長度、必留資訊與禁止推測。

### B. 分類
前面的 `classify_feedback()` 示範互斥標籤與 few-shot 邊界案例。


### C. 改寫：指定讀者、語氣與不可改變的事實


In [ ]:
def rewrite_notice(text, audience="大學生", tone="友善", max_chars=100):
    if max_chars <= 0:
        raise ValueError("max_chars 必須是正整數。")

    instructions = f"""
# Identity
你是校園行政文字編輯。
# Instructions
- 改寫成適合{audience}閱讀的{tone}繁體中文通知。
- 不得更改日期、時間、地點與應繳項目。
- 輸出一個短標題及一段通知，總字數不超過 {max_chars} 字。
"""
    result = ask_ai(f"<original_notice>{text}</original_notice>", instructions)
    if len(result.replace("\n", "")) > max_chars:
        print(f"⚠️ 字數檢查：模型輸出超過 {max_chars} 字，請調整 prompt 或改用結構化流程。")
    return result


notice = "本課程作業二應於10月15日23:59前上傳Moodle，逾期系統關閉，不收email補交。"
print(rewrite_notice(notice, audience="大學生", tone="友善", max_chars=100))


### D. 依據資料問答：限定可使用的 context

這是後續 RAG 的基礎。模型只能根據提供資料回答；資料沒有答案時，必須清楚回報資料不足。


In [ ]:
def answer_from_context(
    question,
    context,
    missing_reply="資料不足，請詢問授課教師。",
):
    instructions = f"""
# Identity
你是課程資訊問答助理。
# Instructions
- 只能根據 `<course_info>` 回答，不可用常識補充規定。
- 無法回答時固定回覆「{missing_reply}」
- 可以回答時先給直接答案，再附簡短依據。
"""
    data = f"<course_info>{context}</course_info><question>{question}</question>"
    return ask_ai(data, instructions)


course_rules = "作業二截止時間為10月15日23:59，繳交平台為Moodle。"
print(answer_from_context("作業二要交到哪裡？", course_rules))
print(answer_from_context("作業二占學期成績幾分？", course_rules))


---

## 8️⃣ Prompt 測試：不要只看一次漂亮答案

至少測試一般案例、邊界／資訊不足案例、容易混淆案例，以及很短、很長或格式異常的輸入。本週先用簡單規則檢查，未來再加入正式 eval。


In [ ]:
classification_tests = [
    {"input": "希望增加更多 Prompt 範例。", "expected": "內容"},
    {"input": "老師講太快。", "expected": "進度"},
    {"input": "教室網路一直斷線。", "expected": "設備"},
    {"input": "希望期末自由組隊。", "expected": "其他"},
    {"input": "投影看不清楚，而且老師切換畫面太快。", "expected": "設備"},
]


def run_classification_tests(test_cases):
    passed = 0
    for index, case in enumerate(test_cases, start=1):
        try:
            actual = classify_feedback(case["input"])
            ok = actual == case["expected"]
        except (RuntimeError, ValueError) as exc:
            actual = f"ERROR: {exc}"
            ok = False
        passed += int(ok)
        print(f"{'✅' if ok else '❌'} Case {index}: expected={case['expected']}, actual={actual}")
    print(f"通過 {passed}/{len(test_cases)} 個案例｜模型：{MODEL}")
    return passed


# 改成 True 會送出 5 次 API 請求。
RUN_BATCH_TESTS = False
if RUN_BATCH_TESTS:
    run_classification_tests(classification_tests)
else:
    print("批次測試尚未執行；確認 API 成本後，將 RUN_BATCH_TESTS 改為 True。")


### 🧪 Prompt 改版紀錄

| 版本 | 修改內容 | 測試結果 | 保留／退回 | 原因 |
|---|---|---:|---|---|
| v1 | 基準規則 |  / 4 |  |  |
| v2 | 增加 few-shot |  / 4 |  |  |
| v3 | 增加邊界案例 |  / 4 |  |  |

一次改一個主要變因並保存測試紀錄，否則很難判斷改善來自哪裡。


---

## 9️⃣ 課堂練習 A：改善客服意見分類器

標籤固定為 `功能問題`、`帳務問題`、`操作疑問`、`其他`。使用 Identity、Instructions、Examples，至少四個 few-shot examples，且只輸出分類名稱。


In [ ]:
SUPPORT_LABELS = ["功能問題", "帳務問題", "操作疑問", "其他"]
SUPPORT_EXAMPLES = [
    ("上傳後畫面一直轉圈。", "功能問題"),
    ("請問如何申請退款？", "帳務問題"),
    ("我要在哪裡修改密碼？", "操作疑問"),
    ("希望增加深色模式。", "其他"),
]

SUPPORT_CLASSIFIER_PROMPT = """
# Identity
你是 SaaS 產品的客服意見分類器。

# Instructions
- 系統故障歸類為「功能問題」；收費、付款、退款歸類為「帳務問題」。
- 不知道如何使用正常功能歸類為「操作疑問」；其餘歸類為「其他」。
- 只輸出分類名稱，不要解釋。
"""


def classify_support_message(message):
    examples = "".join(
        f"<example><input>{text}</input><output>{label}</output></example>\n"
        for text, label in SUPPORT_EXAMPLES
    )
    instructions = SUPPORT_CLASSIFIER_PROMPT + "\n# Examples\n" + examples
    result = ask_ai(f"<support_message>{message}</support_message>", instructions).strip()
    if result not in SUPPORT_LABELS:
        raise ValueError(f"模型輸出不符合分類契約：{result!r}")
    return result


practice_a_inputs = [
    "我的信用卡被扣款兩次。",
    "我找不到匯出報表的按鈕。",
    "謝謝客服快速幫我處理完成！",
]

for message in practice_a_inputs:
    print(message, "→", classify_support_message(message))


---

## 🔟 課堂練習 B：建立學習內容改寫器

完成 `rewrite_for_beginner()`：保留專有名詞並白話解釋、加入生活化比喻、最後提出一題理解題但不給答案，全文 250 字內。


In [ ]:
BEGINNER_REWRITE_PROMPT_TEMPLATE = """
# Identity
你是擅長用白話解釋資訊技術的大學助教。

# Instructions
- 改寫給第一次接觸主題的大學生。
- 保留專有名詞，第一次出現時立即用白話解釋。
- 使用一個生活化比喻，不新增原文無法支持的技術細節。
- 最後提出一題理解檢查題，但不要提供答案。
- 使用繁體中文，全文不超過 {max_chars} 字。

# Output format
## 白話說明
（改寫內容）

## 想一想
（一題理解檢查題）
"""


def rewrite_for_beginner(text, max_chars=250):
    if max_chars <= 0:
        raise ValueError("max_chars 必須是正整數。")
    instructions = BEGINNER_REWRITE_PROMPT_TEMPLATE.format(max_chars=max_chars)
    result = ask_ai(f"<technical_text>{text}</technical_text>", instructions)
    if len(result.replace("\n", "")) > max_chars:
        print(f"⚠️ 字數檢查：模型輸出超過 {max_chars} 字。")
    return result


technical_text = """
Embedding 是把文字轉換成高維度數值向量的技術。語意相近的文字，
其向量在空間中的距離通常較近，因此可以用來進行語意搜尋。
"""

print(rewrite_for_beginner(technical_text, max_chars=250))


---

## 1️⃣1️⃣ 課堂練習 C：依據課程規定問答

完成不亂猜的課程問答 prompt，同時測試有答案與沒有答案的問題。


In [ ]:
policy = """
期中專題採個人製作，於第8週課堂展示。作品需包含簡單操作介面、
至少一次 API 呼叫與錯誤處理。展示時間每人4分鐘。
"""
questions = ["期中專題可以兩人一組嗎？", "每人展示多久？", "期中專題占總成績幾分？"]


def answer_policy_question(
    question,
    missing_reply="資料不足，請詢問授課教師。",
):
    instructions = f"""
# Identity
你是課程規定問答助理。
# Instructions
- 只能根據 `<policy>` 中的文字回答，不得自行推測。
- 若資料沒有答案，固定回覆「{missing_reply}」
- 有答案時，用一至兩句繁體中文直接回答。
"""
    user_input = f"<policy>{policy}</policy><question>{question}</question>"
    return ask_ai(user_input, instructions)


for question in questions:
    print("問題：", question)
    print("回答：", answer_policy_question(question))
    print()


---

## ✅ 完成檢核

- [ ] 固定規則放在 `instructions`，每次資料放在 `input`。
- [ ] 寫出角色、任務、背景、限制與輸出格式。
- [ ] 用 Markdown／XML 區隔 prompt 結構。
- [ ] 為分類任務設計多樣化 few-shot examples。
- [ ] 用參數調整摘要長度、分類標籤或改寫語氣。
- [ ] 用 `compare_prompts()` 對同一輸入比較多個版本。
- [ ] 準備一般、邊界、資訊不足與格式錯誤案例。
- [ ] 說明文字格式限制與 Structured Outputs 的差別。


## 📝 課後任務：Prompt 實驗紀錄

從摘要、分類、改寫、問答選一種功能，繳交：v1 基準 prompt、v2 改良 prompt、至少五筆測試案例（含一筆邊界或衝突案例）、結果比較表、模型名稱，以及 150 字反思。

至少加入一項程式檢查，例如：合法分類、字數上限、必要標題或資料不足固定回覆。

<div style="border-left:5px solid #d93025;padding:10px 14px;background:#fce8e6;">
<font color="#b3261e"><b>安全提醒</b></font>：測試資料不得包含 API key、密碼、學生個資、未公開成績或其他敏感資訊。
</div>


## 📚 延伸閱讀

- OpenAI Prompt engineering：https://developers.openai.com/api/docs/guides/prompt-engineering
- OpenAI Text generation：https://developers.openai.com/api/docs/guides/text

下週將進入：**多輪對話、conversation state 與 Streaming 輸出**。
